# Tratamento da base do Atlas Digital de Desastres

Reproduz em Python (Google Colab) as transformações feitas em SQL para gerar a `BD_Atlas_Final` a partir da base bruta do Atlas Digital.

Etapas:
1. Carga da base bruta
2. Filtro: `Status = 'Reconhecido'` e eventos a partir de 2013
3. Remoção de protocolos duplicados
4. Remoção de registros com `Data_Registro` anterior à `Data_Evento`
5. Padronização dos nomes de municípios (`Mapeamento_Nomes`)
6. Correção de `DH_OUTROS AFETADOS` duplicando `DH_AFETADOS_SECA_ESTIAGEM`
7. Sinalização de danos materiais com valor e sem quantidade
8. Validações e exportação

## 0. Configuração (Google Colab)
Defina abaixo como a base bruta do Atlas Digital (CSV ou Excel baixado em https://atlasdigital.mdr.gov.br/paginas/downloads.xhtml) será carregada:
- `ORIGEM = "upload"`: abre a janela de upload do Colab.
- `ORIGEM = "drive"`: lê de `CAMINHO_DRIVE` no seu Google Drive.

O gabarito (`BD_Atlas_Final.csv`) é opcional; se informado, a última célula compara o resultado com ele.

In [ ]:
ORIGEM = "upload"  # "upload" ou "drive"
CAMINHO_DRIVE = "/content/drive/MyDrive/GrandeDesafio/BD_Atlas_1991_2025_v1.1_2026.08.06_Consolidado.csv"
CAMINHO_GABARITO = "/content/drive/MyDrive/GrandeDesafio/BD_Atlas_Final.csv"  # ou None
ARQ_SAIDA = "BD_Atlas_Final.csv"

In [ ]:
from pathlib import Path

import pandas as pd

if ORIGEM == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ARQ_ENTRADA = Path(CAMINHO_DRIVE)
else:
    from google.colab import files
    enviado = files.upload()
    ARQ_ENTRADA = Path(next(iter(enviado)))

print("Base bruta:", ARQ_ENTRADA)

## 1. Carga

In [ ]:
def le_base(caminho):
    caminho = Path(caminho)
    if caminho.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(caminho, na_values=["NULL"], keep_default_na=False)
    # o CSV oficial do Atlas vem em latin-1 e separado por ";"
    for encoding in ("utf-8", "latin-1"):
        try:
            with open(caminho, encoding=encoding) as f:
                sep = ";" if ";" in f.readline() else ","
            return pd.read_csv(caminho, sep=sep, encoding=encoding, na_values=["NULL"],
                               keep_default_na=False, low_memory=False)
        except UnicodeDecodeError:
            continue


def para_data(col):
    # Excel pode trazer datetime; CSV traz texto dd/mm/aaaa
    if pd.api.types.is_datetime64_any_dtype(col):
        return col
    return pd.to_datetime(col.astype(str).str.strip(), format="%d/%m/%Y", errors="coerce")


df = le_base(ARQ_ENTRADA)
# textos: remove espaços/tabs nas pontas e trata texto vazio como nulo
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip(" \t\r\n").replace("", pd.NA)
df["Data_Evento_dt"] = para_data(df["Data_Evento"])
df["Data_Registro_dt"] = para_data(df["Data_Registro"])
# mantém o formato texto dd/mm/aaaa do gabarito
df["Data_Evento"] = df["Data_Evento_dt"].dt.strftime("%d/%m/%Y")
df["Data_Registro"] = df["Data_Registro_dt"].dt.strftime("%d/%m/%Y")
df["Cod_IBGE_Mun"] = pd.to_numeric(df["Cod_IBGE_Mun"], errors="coerce").astype("Int64")
print(df.shape)
print("Datas não reconhecidas:", df["Data_Evento_dt"].isna().sum(), "/", df["Data_Registro_dt"].isna().sum())
df["Status"].value_counts(dropna=False)

### Checagens iniciais
- Status fora do padrão (`Registro`, `Reconhecido`, `Não reconhecido`)
- Cada `tipologia` associada a uma única `descricao_tipologia`

In [ ]:
status_validos = {"Registro", "Reconhecido", "Não reconhecido"}
print("Status fora do padrão:", (~df["Status"].isin(status_validos)).sum())

tip = df.groupby("tipologia")["descricao_tipologia"].nunique()
print("Tipologias com mais de uma descrição:", (tip > 1).sum())
df.groupby(["tipologia", "descricao_tipologia"]).size().rename("total_eventos").reset_index()

## 2. Filtro: reconhecidos a partir de 2013

In [ ]:
df = df[(df["Status"] == "Reconhecido") & (df["Data_Evento_dt"].dt.year >= 2013)].copy()
print(df.shape)

## 3. Protocolos duplicados
Mantém a primeira ocorrência (equivalente ao `MIN(rowid)` do SQL).

In [ ]:
print("Protocolos duplicados:", df["Protocolo_S2iD"].duplicated().sum())
df = df.drop_duplicates(subset="Protocolo_S2iD", keep="first")
print(df.shape)

## 4. Data de registro anterior à data do evento
Alguns registros têm `Data_Registro` anterior à `Data_Evento`, o que não faz sentido pela lógica do protocolo (e da natureza): primeiro ocorre o evento, depois ele é registrado. Esses registros são removidos.

In [ ]:
datas_invalidas = df["Data_Registro_dt"] < df["Data_Evento_dt"]
display(df.loc[datas_invalidas, ["Protocolo_S2iD", "Nome_Municipio", "Data_Registro", "Data_Evento"]])
df = df[~datas_invalidas].copy()
print(df.shape)

## 5. Código IBGE com mais de um nome de município
O mesmo `Cod_IBGE_Mun` aparece com grafias diferentes (ex.: `` D`Oeste `` / `DOeste`, nomes antigos). Cada código recebe um único nome, definido na tabela `Mapeamento_Nomes`.

In [ ]:
variacoes = df.groupby("Cod_IBGE_Mun")["Nome_Municipio"].agg(["nunique", lambda s: ", ".join(sorted(s.unique()))])
variacoes.columns = ["variacoes_nome", "nomes_encontrados"]
variacoes[variacoes["variacoes_nome"] > 1]

In [ ]:
MAPEAMENTO_NOMES = {
    1100015: "Alta Floresta D'Oeste",
    1100098: "Espigão D'Oeste",
    1100130: "Machadinho D'Oeste",
    1100148: "Nova Brasilândia D'Oeste",
    1100296: "Santa Luzia D'Oeste",
    1502954: "Eldorado dos Carajás",
    1505551: "Pau D'Arco",
    1506500: "Santa Izabel do Pará",
    2201176: "Barra D'Alcântara",
    2306306: "Itapajé",
    2401305: "Augusto Severo",
    2405306: "Boa Saúde",
    2406205: "Lagoa D'Anta",
    2408409: "Olho D'Água do Borges",
    2508703: "Mãe D'Água",
    2510402: "Olho D'Água",
    2512606: "Quixabá",
    2515401: "São Vicente do Seridó",
    2606903: "Iguaracy",
    2613107: "São Caetano",
    2613800: "São Vicente Férrer",
    2705705: "Olho D'Água das Flores",
    2705804: "Olho D'Água do Casado",
    2709004: "Tanque D'Arca",
    2917334: "Iuiú",
    2922250: "Muquém de São Francisco",
    2928505: "Santa Terezinha",
    3108909: "Brazópolis",
    3145455: "Olhos-D'Água",
    3200706: "Atílio Vivácqua",
    3545803: "Santa Bárbara D'Oeste",
    3550001: "São Luiz do Paraitinga",
    4107157: "Diamante D'Oeste",
    4111209: "Itapejara D'Oeste",
    4119004: "Pérola D'Oeste",
    4121356: "Rancho Alegre D'Oeste",
    4125209: "São Jorge D'Oeste",
    4206702: "Herval D'Oeste",
    4209607: "Lauro Müller",
    4216057: "São Cristóvão do Sul",
    4315503: "Restinga Sêca",
    4317103: "Sant'Ana do Livramento",
    4322855: "Vespasiano Corrêa",
    4323770: "Westfália",
    5103361: "Conquista D'Oeste",
    5103809: "Figueirópolis D'Oeste",
    5103957: "Glória D'Oeste",
    5105234: "Lambari D'Oeste",
    5105622: "Mirassol D'Oeste",
    5107008: "Poxoréu",
    5220009: "São João D'Aliança",
}

mapeado = df["Cod_IBGE_Mun"].isin(MAPEAMENTO_NOMES)
novo_nome = df.loc[mapeado, "Cod_IBGE_Mun"].map(MAPEAMENTO_NOMES)
print("Registros renomeados:", (df.loc[mapeado, "Nome_Municipio"] != novo_nome).sum())
df.loc[mapeado, "Nome_Municipio"] = novo_nome

restantes = df.groupby("Cod_IBGE_Mun")["Nome_Municipio"].nunique()
print("Códigos com mais de um nome após correção:", (restantes > 1).sum())

## 6. Estiagem: `DH_OUTROS AFETADOS` repete `DH_AFETADOS_SECA_ESTIAGEM`
Na estiagem, "outros afetados" repete o mesmo número de "afetados por seca". O total humano já soma a seca e não soma "outros"; quem somar as duas colunas conta a mesma gente duas vezes.

Zeramos `DH_OUTROS AFETADOS` onde ele repete a seca. O total humano não muda, porque essa coluna não entra na conta.

In [ ]:
DH_COLS = ["DH_MORTOS", "DH_FERIDOS", "DH_ENFERMOS", "DH_DESABRIGADOS", "DH_DESALOJADOS",
           "DH_DESAPARECIDOS", "DH_AFETADOS_SECA_ESTIAGEM", "DH_OUTROS AFETADOS"]

duplicado = (df["DH_OUTROS AFETADOS"] == df["DH_AFETADOS_SECA_ESTIAGEM"]) & (df["DH_OUTROS AFETADOS"].fillna(0) > 0)
display(df.loc[duplicado, ["Protocolo_S2iD", "descricao_tipologia", "DH_OUTROS AFETADOS",
                           "DH_AFETADOS_SECA_ESTIAGEM", "DH_total_danos_humanos_diretos"]].head())
print("Registros corrigidos:", duplicado.sum())
df.loc[duplicado, "DH_OUTROS AFETADOS"] = 0

## 7. Dano material com valor e quantidade zerada
Há valor de dano material preenchido com quantidade danificada e destruída zerada, e o total material inclui esse valor.

A quantidade fica na base e fora das análises; o valor permanece. A sinalização é uma coluna nova, `Sinal_DM_Valor_Sem_Qtd`, igual a 1 só nas linhas com valor e quantidade zerada.

In [ ]:
DM_CATEGORIAS = ["Uni Habita", "Inst Saúde", "Inst Ensino", "Inst Serviços", "Inst Comuni", "Obras de Infra"]

sinal = pd.Series(False, index=df.index)
for cat in DM_CATEGORIAS:
    qtd = df[f"DM_{cat} Danificadas"].fillna(0) + df[f"DM_{cat} Destruidas"].fillna(0)
    sinal |= (df[f"DM_{cat} Valor"].fillna(0) > 0) & (qtd == 0)
df["Sinal_DM_Valor_Sem_Qtd"] = sinal.astype(int)
df["Sinal_DM_Valor_Sem_Qtd"].value_counts()

## 8. Validações finais
Mesmas checagens do relatório de contagem do SQL (tolerância de 1 para arredondamento).

In [ ]:
DM_VALORES = [f"DM_{c} Valor" for c in DM_CATEGORIAS]

def divergencias(partes, total):
    return ((df[partes].fillna(0).sum(axis=1) - df[total].fillna(0)).abs() > 1).sum()

relatorio = {
    "Status_Corrigidos": (~df["Status"].isin(status_validos)).sum(),
    "Duplicatas": df["Protocolo_S2iD"].duplicated().sum(),
    "Municipios_Corrigidos": (df.groupby("Cod_IBGE_Mun")["Nome_Municipio"].nunique() > 1).sum(),
    "DH_Divergencias": divergencias(DH_COLS, "DH_total_danos_humanos_diretos"),
    "DM_Divergencias": divergencias(DM_VALORES, "DM_total_danos_materiais"),
    "PE_Divergencias": divergencias(["PEPL_total_publico", "PEPR_total_privado"], "PE_PLePR"),
}
pd.Series(relatorio, name="quantidade")

## 9. Exportação

In [ ]:
df_final = df.drop(columns=["Data_Evento_dt", "Data_Registro_dt"])
df_final.to_csv(ARQ_SAIDA, index=False)
print(df_final.shape, "->", ARQ_SAIDA)

from google.colab import files
files.download(ARQ_SAIDA)

### Conferência com o gabarito (base gerada via SQL)

In [ ]:
if CAMINHO_GABARITO and Path(CAMINHO_GABARITO).exists():
    ref = pd.read_csv(CAMINHO_GABARITO, low_memory=False).set_index("Protocolo_S2iD")
    novo = pd.read_csv(ARQ_SAIDA, low_memory=False).set_index("Protocolo_S2iD")
    print("Mesmos protocolos:", set(ref.index) == set(novo.index))
    print("Mesmas colunas:", list(ref.columns) == list(novo.columns))
    if set(ref.index) == set(novo.index):
        diff = ref.compare(novo.loc[ref.index])
        print("Células diferentes:", int(diff.notna().sum().sum() // 2))
        display(diff.head(20))
else:
    print("Gabarito não informado; conferência ignorada.")